In [1]:
%pip install -q groq tenacity sentence-transformers faiss-cpu tqdm pandas numpy


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
pip install together

  Using cached docutils-0.22.4-py3-none-any.whl.metadata (15 kB)
Using cached docutils-0.22.4-py3-none-any.whl (633 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [together] 10/11 [together]

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
import os
from getpass import getpass

if "TOGETHER_API_KEY" not in os.environ:
    os.environ["TOGETHER_API_KEY"] = getpass("Enter TOGETHER_API_KEY: ")

print("Together API key loaded:", "TOGETHER_API_KEY" in os.environ)

Together API key loaded: True


In [2]:
import json
import time
import os
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from groq import Groq

In [3]:

INDEX_DIR = Path("indexes")

# Load the FAISS index file (the actual vectors)
index = faiss.read_index(str(INDEX_DIR / "recursive_128.faiss"))

# Load the chunk metadata (doc_id, text for each vector)
with open(INDEX_DIR / "recursive_128_chunks.json") as f:
    chunks = json.load(f)

print(f"Index loaded: {index.ntotal} vectors")
print(f"Chunks loaded: {len(chunks)}")
print(f"Match: {index.ntotal == len(chunks)}")

# Quick sanity check on chunk structure
print(f"\nSample chunk keys: {chunks[0].keys()}")
print(f"Sample doc_id: {chunks[0]['doc_id']}")
print(f"Sample text: {chunks[0]['text'][:100]}...")

Index loaded: 18638 vectors
Chunks loaded: 18638
Match: True

Sample chunk keys: dict_keys(['chunk_id', 'doc_id', 'chunk_index', 'chunk_type', 'text'])
Sample doc_id: 4983
Sample text: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...


In [25]:
# MUST use the exact same model and prefix as retrieval eval
# Changing either would break consistency with your measured Hit Rate

QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

print("Loading embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"Loaded. Embedding dim: {embed_model.get_sentence_embedding_dimension()}")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded. Embedding dim: 384


/var/folders/qk/phwdhr193ssd2dntzk54h7vh0000gn/T/ipykernel_293/3256792643.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded. Embedding dim: {embed_model.get_sentence_embedding_dimension()}")


In [26]:
# Load TRAIN claims — 957 claims
# These are for teacher generation and distillation
# DO NOT use claims_val.json here — that's locked for final evaluation

with open("claims_train.json") as f:
    claims = json.load(f)

print(f"Loaded {len(claims)} train claims")
print(f"\nSample claim: {claims[0]['claim']}")
print(f"Ground truth label: {claims[0]['evidence_label']}")
print(f"Ground truth doc_id: {claims[0]['evidence_doc_id']}")

Loaded 957 train claims

Sample claim: 1 in 5 million in UK have abnormal PrP positivity.
Ground truth label: CONTRADICT
Ground truth doc_id: 13734012


In [29]:
from together import Together

client = Together(api_key=os.environ.get("TOGETHER_API_KEY"))

# Test connection
test = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    messages=[{"role": "user", "content": "Reply with one word: ready"}],
    max_tokens=5
)
print(f"Together AI connected. Response: {test.choices[0].message.content}")

Together AI connected. Response: Ready


In [30]:
# Structured format is critical here
# The label parser depends on "LABEL:" appearing at the start of a line
# Don't change the format — changing it breaks parsing downstream

TEACHER_PROMPT = """You are an expert biomedical fact-checker.

Given a scientific CLAIM and EVIDENCE retrieved from research paper abstracts, determine whether the evidence supports or contradicts the claim.

CLAIM: {claim}

EVIDENCE:
{evidence}

Respond in this exact format and nothing else:
LABEL: [SUPPORT or CONTRADICT or NOT_ENOUGH_INFO]
REASONING: [2-3 sentences explaining your decision, citing specific parts of the evidence]
CONFIDENCE: [HIGH or MEDIUM or LOW]"""

In [31]:
def retrieve_evidence(claim_text, top_k=5):
    """
    Retrieves top-k chunks using FAISS.
    Uses identical logic to rag_retrieval_eval_fixed.py so
    teacher retrieval is consistent with your measured metrics.
    
    Returns:
        evidence_text  — formatted string for LLM prompt
        retrieved_docs — list of dicts with doc_id, score, text
    """
    # Prefix query exactly as done in retrieval eval
    prefixed = QUERY_INSTRUCTION + claim_text
    
    # Embed the query
    query_embedding = embed_model.encode(
        [prefixed],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")
    
    # Over-retrieve then deduplicate by doc_id
    # Same logic as evaluate() in rag_retrieval_eval_fixed.py
    search_k = min(top_k * 5, index.ntotal)
    scores, indices = index.search(query_embedding, search_k)
    
    # Deduplicate — keep best-ranked chunk per document
    seen_docs = {}
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]
        doc_id = str(chunk["doc_id"])
        if doc_id not in seen_docs:
            seen_docs[doc_id] = {
                "doc_id": chunk["doc_id"],
                "score": float(score),
                "text": chunk["text"],
                "chunk_id": chunk["chunk_id"]
            }
        if len(seen_docs) >= top_k:
            break
    
    retrieved_docs = list(seen_docs.values())
    
    # Format as numbered evidence for the prompt
    evidence_text = ""
    for i, doc in enumerate(retrieved_docs):
        evidence_text += f"[{i+1}] {doc['text']}\n\n"
    
    return evidence_text.strip(), retrieved_docs

In [41]:
# # Replace Groq client with Together
# from together import Together
# client = Together(api_key="TOGETHER_API_KEY")

def call_teacher(claim_text, evidence_text):
    prompt = TEACHER_PROMPT.format(
        claim=claim_text,
        evidence=evidence_text
    )
    try:
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"    API error: {e}")
        return None

In [42]:
def parse_label(response_text):
    if not response_text:
        return "PARSE_ERROR"
    
    for line in response_text.strip().split("\n"):
        if line.startswith("LABEL:"):
            label = line.replace("LABEL:", "").strip().upper()
            if "SUPPORT" in label:
                return "SUPPORT"
            elif "REFUTE" in label or "CONTRADICT" in label:
                return "CONTRADICT"   # match SciFact's label
            elif "NOT_ENOUGH" in label or "INSUFFICIENT" in label:
                return "NOT_ENOUGH_INFO"
    
    return "PARSE_ERROR"


def parse_confidence(response_text):
    """Extracts CONFIDENCE from structured teacher response."""
    if not response_text:
        return "UNKNOWN"
    
    for line in response_text.strip().split("\n"):
        if line.startswith("CONFIDENCE:"):
            conf = line.replace("CONFIDENCE:", "").strip().upper()
            if "HIGH" in conf:
                return "HIGH"
            elif "MEDIUM" in conf or "MED" in conf:
                return "MEDIUM"
            elif "LOW" in conf:
                return "LOW"
    
    return "UNKNOWN"

In [44]:
import os
os.environ["TOGETHER_API_KEY"] = "tgp_v1_JlYqSifBCDdEqoNhWKJpkN_ylZjAar-UH4sN2mG7yDc"

from together import Together
client = Together(api_key=os.environ.get("TOGETHER_API_KEY"))

# Test
test = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    messages=[{"role": "user", "content": "Reply with one word: ready"}],
    max_tokens=5
)
print(f"Connected: {test.choices[0].message.content}")

Connected: Ready


In [45]:
# ALWAYS do this before running 957 claims
# Catches any issues with retrieval, prompt format, or parsing
# before you waste API quota

test_claim = claims[0]

print(f"Claim: {test_claim['claim']}")
print(f"Ground truth: {test_claim['evidence_label']}")
print(f"Ground truth doc: {test_claim['evidence_doc_id']}")
print()

# Test retrieval
evidence_text, retrieved_docs = retrieve_evidence(test_claim["claim"])
print(f"Retrieved {len(retrieved_docs)} docs:")
for doc in retrieved_docs:
    print(f"  doc_id={doc['doc_id']}  score={doc['score']:.4f}")
print()

# Test teacher call
print("Calling teacher model...")
response = call_teacher(test_claim["claim"], evidence_text)
print(f"\nTeacher response:\n{response}")
print()

# Test parsing
print(f"Parsed label:      {parse_label(response)}")
print(f"Parsed confidence: {parse_confidence(response)}")
print(f"Ground truth:      {test_claim['evidence_label']}")
print(f"Correct:           {parse_label(response) == test_claim['evidence_label']}")

Claim: 1 in 5 million in UK have abnormal PrP positivity.
Ground truth: CONTRADICT
Ground truth doc: 13734012

Retrieved 5 docs:
  doc_id=13734012  score=0.7854
  doc_id=27711043  score=0.7123
  doc_id=19945096  score=0.6991
  doc_id=841371  score=0.6832
  doc_id=5850219  score=0.6794

Calling teacher model...

Teacher response:
LABEL: CONTRADICT
REASONING: The evidence from [1] directly contradicts the claim, as it reports an overall prevalence of 493 per million population with abnormal PrP positivity, which is significantly higher than the claimed 1 in 5 million. This study provides a specific and relevant measurement of the prevalence, making it a strong contradiction to the claim. The confidence interval provided in [1] also does not include the claimed rate, further supporting the contradiction.
CONFIDENCE: HIGH

Parsed label:      CONTRADICT
Parsed confidence: HIGH
Ground truth:      CONTRADICT
Correct:           True


In [46]:
os.makedirs("logs", exist_ok=True)

# Load checkpoint from previous Groq run (100 claims already done)
checkpoint_path = "logs/teacher_logs_checkpoint.json"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        teacher_logs = json.load(f)
    completed_ids = {log["claim_id"] for log in teacher_logs}
    remaining_claims = [c for c in claims if c["id"] not in completed_ids]
    print(f"Resuming from checkpoint")
    print(f"Already completed: {len(teacher_logs)}")
    print(f"Remaining:         {len(remaining_claims)}")
else:
    teacher_logs = []
    completed_ids = set()
    remaining_claims = claims
    print(f"No checkpoint found, starting fresh")
    print(f"Total claims: {len(remaining_claims)}")

api_errors = 0
parse_errors = 0
start_time = time.time()

print(f"\nStarting generation...\n")

for i, claim in enumerate(remaining_claims):

    if i % 50 == 0 and i > 0:
        elapsed = (time.time() - start_time) / 60
        rate = i / max(elapsed, 0.1)
        eta = (len(remaining_claims) - i) / max(rate, 0.1)
        print(f"  [{i}/{len(remaining_claims)}] {elapsed:.1f} min elapsed, ~{eta:.0f} min remaining | errors: {api_errors} API, {parse_errors} parse")

    evidence_text, retrieved_docs = retrieve_evidence(claim["claim"])
    response = call_teacher(claim["claim"], evidence_text)

    if response is None:
        api_errors += 1
        time.sleep(2)
        continue

    teacher_label = parse_label(response)
    teacher_confidence = parse_confidence(response)

    if teacher_label == "PARSE_ERROR":
        parse_errors += 1

    teacher_logs.append({
        "claim_id":           claim["id"],
        "claim":              claim["claim"],
        "ground_truth_doc":   claim["evidence_doc_id"],
        "ground_truth_label": claim["evidence_label"],
        "retrieved_docs":     retrieved_docs,
        "evidence_text":      evidence_text,
        "teacher_response":   response,
        "teacher_label":      teacher_label,
        "teacher_confidence": teacher_confidence,
        "correct":            teacher_label == claim["evidence_label"]
    })

    if (i + 1) % 100 == 0:
        with open(checkpoint_path, "w") as f:
            json.dump(teacher_logs, f)
        print(f"  Checkpoint saved: {len(teacher_logs)} total")

    time.sleep(0.3)

# Final save
with open("logs/teacher_logs.json", "w") as f:
    json.dump(teacher_logs, f, indent=2)

total_time = (time.time() - start_time) / 60
print(f"\nDone in {total_time:.1f} minutes")
print(f"Total logged: {len(teacher_logs)}/957")
print(f"API errors:   {api_errors}")
print(f"Parse errors: {parse_errors}")

Resuming from checkpoint
Already completed: 100
Remaining:         857

Starting generation...

  [50/857] 3.6 min elapsed, ~58 min remaining | errors: 0 API, 0 parse
  Checkpoint saved: 200 total
  [100/857] 6.0 min elapsed, ~46 min remaining | errors: 0 API, 0 parse
  [150/857] 9.5 min elapsed, ~45 min remaining | errors: 0 API, 0 parse
  Checkpoint saved: 300 total
  [200/857] 13.4 min elapsed, ~44 min remaining | errors: 0 API, 0 parse
  [250/857] 16.7 min elapsed, ~41 min remaining | errors: 0 API, 0 parse
  Checkpoint saved: 400 total
  [300/857] 19.4 min elapsed, ~36 min remaining | errors: 0 API, 0 parse
  [350/857] 22.4 min elapsed, ~32 min remaining | errors: 0 API, 0 parse
  Checkpoint saved: 500 total
  [400/857] 25.1 min elapsed, ~29 min remaining | errors: 0 API, 0 parse
  [450/857] 27.8 min elapsed, ~25 min remaining | errors: 0 API, 0 parse
  Checkpoint saved: 600 total
  [500/857] 30.9 min elapsed, ~22 min remaining | errors: 0 API, 0 parse
  [550/857] 33.6 min elapsed

In [ ]:
with open("logs/teacher_logs.json") as f:
    teacher_logs = json.load(f)

from collections import Counter

total = len(teacher_logs)
correct = sum(1 for log in teacher_logs if log["correct"])
parse_err = sum(1 for log in teacher_logs if log["teacher_label"] == "PARSE_ERROR")

print(f"Total logged:  {total}")
print(f"Correct:       {correct} ({correct/total:.1%})")
print(f"Wrong:         {total - correct - parse_err} ({(total-correct-parse_err)/total:.1%})")
print(f"Parse errors:  {parse_err} ({parse_err/total:.1%})")
print()

# Label distributions — compare teacher vs ground truth
teacher_dist = Counter(log["teacher_label"] for log in teacher_logs)
gt_dist = Counter(log["ground_truth_label"] for log in teacher_logs)

print(f"{'Label':<20} {'Teacher':>10} {'Ground Truth':>14}")
print("-" * 46)
all_labels = sorted(set(list(teacher_dist.keys()) + list(gt_dist.keys())))
for label in all_labels:
    t = teacher_dist.get(label, 0)
    g = gt_dist.get(label, 0)
    print(f"{label:<20} {t:>6} ({t/total:.1%}) {g:>6} ({g/total:.1%})")

In [ ]:
# Convert teacher logs into training pairs for distillation
# Person B needs logs/training_data.json to start Day 8 on Kaggle

training_data = []

for log in teacher_logs:
    # Skip malformed responses — don't train on bad data
    if log["teacher_label"] == "PARSE_ERROR":
        continue
    
    training_data.append({
        "claim_id":    log["claim_id"],
        "instruction": f"""You are a biomedical fact-checker.

CLAIM: {log['claim']}

EVIDENCE:
{log['evidence_text']}

Analyze the claim based on the evidence above.""",
        "output": log["teacher_response"]
    })

with open("logs/training_data.json", "w") as f:
    json.dump(training_data, f, indent=2)

print(f"Saved {len(training_data)} training examples")
print(f"Skipped {len(teacher_logs) - len(training_data)} parse errors")

In [24]:
import json
try:
    with open("logs/teacher_logs_checkpoint.json") as f:
        logs = json.load(f)
    print(f"Saved so far: {len(logs)} claims")
except:
    print("No checkpoint found")

Saved so far: 100 claims
